## Pull Raw Data

#### Load Libraries

In [15]:
import pandas as pd 
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from smart_open import open as s_open
import zipfile
import io 
import duckdb
from dotenv import load_dotenv
import os

#### Get zip file for test

In [16]:
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

bucket = 'tripdata'

#pull objects from s3 bucket
objects = s3.list_objects_v2(Bucket=bucket)
objects_sort = sorted(objects['Contents'], key=lambda x: x['LastModified'], reverse=True)

#check most recent obect
key = objects_sort[0]['Key']
    
file = s3.get_object(Bucket=bucket, Key=key)
data = file['Body'].read()


In [17]:
# zip contents and size 

print(f"\n=== {key} ==")
with zipfile.ZipFile(io.BytesIO(data)) as zf:
    for info in zf.infolist():
        print(f"  {info.filename}  ({info.file_size/1_000_000:,.1f} MB)")


=== JC-202604-citibike-tripdata.zip ==
  JC-202604-citibike-tripdata.csv  (17.8 MB)


#### Transfer to dataframe

In [10]:
with zipfile.ZipFile(io.BytesIO(data)) as zf:
    csv_name = next(n for n in zf.namelist() if n.endswith('.csv'))
    with zf.open(csv_name) as f:
        df = pd.read_csv(f)

df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,558250BE9BDDEF62,classic_bike,2026-04-08 11:01:58.516,2026-04-08 11:15:28.078,City Hall,JC003,Southwest Park - Jackson St & Observer Hwy,HB401,40.717732,-74.043845,40.737551,-74.041664,member
1,DE08A3A0DC829851,electric_bike,2026-04-04 14:28:31.751,2026-04-04 14:31:34.921,6 St & Grand St,HB302,Willow Ave & 12 St,HB505,40.744398,-74.034501,40.751867,-74.030377,member
2,B0434D0A2865B3E2,electric_bike,2026-04-27 18:25:42.870,2026-04-27 18:30:55.446,6 St & Grand St,HB302,Southwest Park - Jackson St & Observer Hwy,HB401,40.744398,-74.034501,40.737551,-74.041664,casual
3,1D805A4835F0A8AB,classic_bike,2026-04-19 16:59:12.513,2026-04-19 17:04:03.409,6 St & Grand St,HB302,Willow Ave & 12 St,HB505,40.744398,-74.034501,40.751867,-74.030377,casual
4,854CE9A4A8705F8F,electric_bike,2026-04-27 18:46:25.717,2026-04-27 18:50:36.255,6 St & Grand St,HB302,Willow Ave & 12 St,HB505,40.744398,-74.034501,40.751867,-74.030377,casual


#### Connect and write raw data to Database

In [ ]:
load_dotenv()

token = os.environ['MOTHERDUCK_TOKEN']
con = duckdb.connect(f'md:citi_bike?motherduck_token={token}')


In [ ]:
raw_schema = 'raw'
table = f"{raw_schema}.trips_test"
period = "APR_2026"

con.sql("CREATE SCHEMA IF NOT EXISTS raw")
    
con.sql(f"""
        CREATE OR REPLACE TABLE {table} AS
        SELECT 
            *
            , '{period}' AS source_period
        FROM 
            df
    """)